[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/03_grouped_query_attention.ipynb)


# Workshop: Building Gemma 3 from Scratch
## Notebook 3: Grouped Query Attention (GQA)

**Estimated Time: 15 minutes**

Grouped Query Attention (GQA), is arguably one of the most important architectural optimizations in modern LLMs (used by Gemma, Llama 3, Mistral, etc.) for reducing memory bottlenecks during inference.

Standard Multi-Head Attention (MHA) gives every Query head its own Key and Value head. This is memory-intensive for large models. **Grouped Query Attention (GQA)** optimizes this by sharing one K and V head among multiple Query heads.

## Learning Objectives:
1. Compare Multi-Head Attention (MHA), Multi-Query Attention (MQA), and Grouped Query Attention (GQA).
2. Understand the efficiency gains of GQA.
3. Implement the tensor repetition logic used in GQA using PyTorch ```repeat_interleave``` so that standard dot-product math still works.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

batch_size = 1
seq_len = 8
num_heads = 8
num_kv_groups = 2
head_dim = 96  # Gemma 3: 768 / 8 = 96

group_size = num_heads // num_kv_groups  # = 4 for Gemma 3 (8/2)
print(f"Each KV head will be shared by {group_size} Query heads.")

Python version: 3.12.0 (main, Oct  2 2023, 20:56:14) [Clang 16.0.3 ]
PyTorch version: 2.12.0
Each KV head will be shared by 4 Query heads.


## 1. MHA vs MQA vs GQA

| Architecture   | Q heads | K heads | V heads | KV-Cache size |
|----------------|---------|---------|---------|---------------|
| **MHA**        | 8       | 8       | 8       | 8x            |
| **MQA**        | 8       | 1       | 1       | 1x            |
| **GQA** (Gemma 3) | 8    | 2       | 2       | 2x            |

1. Multi-Head Attention (MHA): Every Query head gets its own KV head. (8 Q, 8 KV). Great quality, terrible memory footprint.

2. Multi-Query Attention (MQA): All 8 Query heads share exactly 1 KV head. (8 Q, 1 KV). Incredible memory savings, but noticeably degrades reasoning quality.

3. Grouped Query Attention (GQA): The Gemma 3 approach. We divide the 8 Query heads into "groups" (Gemma 3 uses 2 groups). Each group of 4 Query heads shares 1 KV head. (8 Q, 2 KV). Retains the quality of MHA but saves 75% of the memory!

Gemma 3 uses **GQA with 2 KV groups** -- a highly optimized sweet spot between MHA's quality and MQA's speed.

## 2. Generating Q, K, V with Groups

In [2]:
# Shape: (batch, num_heads/groups, seq_len, head_dim)
Q = torch.randn(batch_size, num_heads, seq_len, head_dim)
K = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)
V = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)

print(f"Q heads: {Q.shape[1]}")
print(f"K heads: {K.shape[1]}")
print(f"V heads: {V.shape[1]}")
print(f"KV-Cache saved: {(1 - num_kv_groups / num_heads) * 100:.0f}% vs MHA")

Q heads: 8
K heads: 2
V heads: 2
KV-Cache saved: 75% vs MHA


## 👥 Grouped Query Attention (GQA): The Repetition Trick

Gemma 3 uses **Grouped Query Attention (GQA)** to save memory. In our model, we project **8 Query heads** but only **2 Key/Value heads**. The group size is $8 / 2 = 4$, meaning each KV group is shared by 4 Query heads.

In PyTorch, you cannot multiply an 8-head Query tensor with a 2-head Key tensor. The dimensions must match.

### 🧮 The Repetition Trick (`repeat_interleave`):
To compute standard attention, we must align the Query head count with the Key/Value head count. We expand the groups by repeating them along the head dimension `dim=1`:
- **Input Key shape**: `(batch, 2, seq_len, head_dim)`
- **Repetition**: `K.repeat_interleave(4, dim=1)` repeats each KV head 4 times consecutively:
  $$\text{Head } 0, 1 \quad \rightarrow \quad \underbrace{\text{Head } 0, \text{Head } 0, \text{Head } 0, \text{Head } 0}_{\text{Group 1}}, \quad \underbrace{\text{Head } 1, \text{Head } 1, \text{Head } 1, \text{Head } 1}_{\text{Group 2}}$$
- **Output Key shape**: `(batch, 8, seq_len, head_dim)`

Now, the Key tensor has 8 heads, perfectly aligning with the 8 Query heads for the matrix multiplication! (Note for advanced students: Production implementations like FlashAttention do this expansion implicitly in the GPU's SRAM to save bandwidth, but repeat_interleave is the correct PyTorch equivalent).

In [3]:
# Expand K and V to match Q's number of heads
K_expanded = K.repeat_interleave(group_size, dim=1)
V_expanded = V.repeat_interleave(group_size, dim=1)

print(f"Q heads:     {Q.shape[1]}")
print(f"Expanded K:  {K_expanded.shape[1]}")
print(f"Expanded V:  {V_expanded.shape[1]}")
assert K_expanded.shape[1] == Q.shape[1]
print("OK: K and V successfully expanded to match Q!")

Q heads:     8
Expanded K:  8
Expanded V:  8
OK: K and V successfully expanded to match Q!


## 4. GQA Attention with QK-Norm (Gemma 3)

In [4]:
# QK-Norm attention with GQA
Q_norm = F.normalize(Q, p=2, dim=-1)
K_norm = F.normalize(K_expanded, p=2, dim=-1)

scores = torch.matmul(Q_norm, K_norm.transpose(-2, -1)) / math.sqrt(head_dim)
attention_weights = F.softmax(scores, dim=-1)

output = torch.matmul(attention_weights, V_expanded)
print(f"Attention output shape: {output.shape}")

Attention output shape: torch.Size([1, 8, 8, 96])


## 5. Why GQA?

In inference, we store K and V in a **KV Cache**. By using fewer KV heads, we drastically reduce the memory footprint of the cache, allowing for larger batch sizes and longer contexts.

For Gemma 3 with 2 KV groups:
- KV-cache memory = 2/8 = 25% of full MHA
- Combined with 5:1 local/global layers: total KV reduction < 15%

## 🏗️ Assembling the complete PyTorch GQA Layer

Here we implement the full `SimpleGQA` PyTorch module. It integrates linear head projections, GQA group splitting, group expansion, QK-Norm attention, and final linear output projection.

### 📈 Step-by-Step Operations inside `forward(x)`:
1. **Linear Projection**: Input embedding $x$ of shape `(B, T, C)` is projected into Query, Key, and Value spaces. Key and Value are projected to a smaller size since they only have 2 heads.
2. **Dimension Reshaping**: Tensors are reshaped and transposed to `(batch_size, num_heads, seq_len, head_dim)`.
3. **Repetition Expansion**: Key and Value are expanded along `dim=1` by a factor of `group_size = 4` to match Query's 8 heads.
4. **QK-Norm & Attention**: Q and K are L2-normalized along `dim=-1` before computing scaled dot-products.
5. **Output Projection**: Head outputs are concatenated back to the hidden size `dim=768` and projected through `out_proj`.

In [ ]:
class SimpleGQA(nn.Module):
    def __init__(self, d_in, n_heads, n_kv_groups, h_dim):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.h_dim = h_dim
        self.group_size = n_heads // n_kv_groups

        # GQA projects to fewer K/V dimensions
        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)

    def rms_norm(self, tensor, eps=1e-6):
        """Simple RMSNorm without learnable weights for QK-Norm."""
        return tensor * torch.rsqrt(tensor.pow(2).mean(-1, keepdim=True) + eps)

    def forward(self, x):
        B, T, C = x.shape

        # 1. Project
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)

        # 2. Expand K, V groups to match Q heads
        # TODO: Implement exxpansion here!
        k = ...
        v = ...

        if k is Ellipsis or v is Ellipsis:
            raise NotImplementedError("Implement K, V groups expansion before continuing!")

        # 3. QK-Norm attention (Gemma 3)
        q_norm = self.rms_norm(q)
        k_norm = self.rms_norm(k)
        scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)

        # 4. Reshape and project out
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)


# Test your implementation
d_in, n_h, n_kv, h_d = 96, 8, 2, 96
model = SimpleGQA(d_in, n_h, n_kv, h_d)
x = torch.randn(1, 5, d_in)
output = model(x)
print(f"OK: Success! GQA Output shape: {output.shape}")
assert output.shape == x.shape

AttributeError: 'ellipsis' object has no attribute 'pow'


<details>
<summary><b>Click to see solution</b></summary>

```python
class SimpleGQA(nn.Module):
    def __init__(self, d_in, n_heads, n_kv_groups, h_dim):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.h_dim = h_dim
        self.group_size = n_heads // n_kv_groups

        # GQA projects to fewer K/V dimensions
        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)

    def rms_norm(self, tensor, eps=1e-6):
        """Simple RMSNorm without learnable weights for QK-Norm."""
        return tensor * torch.rsqrt(tensor.pow(2).mean(-1, keepdim=True) + eps)

    def forward(self, x):
        B, T, C = x.shape

        # 1. Project
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)

        # 2. Expand K, V groups to match Q heads
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        # 3. QK-Norm attention (Gemma 3)
        q_norm = self.rms_norm(q)
        k_norm = self.rms_norm(k)
        scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)

        # 4. Reshape and project out
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)
```
